# Question 1: Multilayer Perceptron (MLP)

The goal of this exercise is to design, implement, and analyze a pricing prediction model for the Cars Datasets 2025.

## Section 1: Network Architecture

In this section, we define the architecture of our MLP model according to the assignment requirements. The architecture consists of an input layer, two hidden layers with ReLU activation, and a linear output layer.

Architecture Specifications:

    Input Layer: Dimensions must match the number of features after preprocessing.

    Hidden Layer 1: 64 neurons with ReLU activation function.

    Hidden Layer 2: 32 neurons with ReLU activation function.

    Output Layer: 1 neuron with Linear activation (for regression).

General Requirements:

    Data Splitting: Data must be divided into 70% Training, 15% Validation, and 15% Testing sets.

    Pre-processing: Features must be standardized, and missing values must be handled appropriately.

    Reproducibility: A specific random_state must be used for data splitting.

In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import re

# -----------------------------
# 1. Load Dataset
# -----------------------------
df = pd.read_csv('CarsDatasets.csv')

# -----------------------------
# 2. Clean Price Column
# -----------------------------
import re
import numpy as np

def clean_price(price):
    """
    Extract all numbers from a messy price string and return their mean.
    
    Examples:
    "$12,000-$15,000" -> [12000, 15000] -> 13500
    "55000 / 65000"   -> [55000, 65000] -> 60000
    "$1,100,000"     -> [1100000] -> 1100000
    """
    if pd.isna(price):
        return np.nan
    
    # Extract all numbers
    numbers = re.findall(r'\d+', price)
    
    if len(numbers) == 0:
        return np.nan
    
    numbers = [float(n) for n in numbers]
    return np.mean(numbers)


df['Cars Prices'] = df['Cars Prices'].apply(clean_price)

# -----------------------------
# 3. Basic Preprocessing
# -----------------------------
def preprocess_data(df):
    df = df.copy()
    df = df.fillna(df.median(numeric_only=True))
    return df

df = preprocess_data(df)

# -----------------------------
# 4. Split Features and Target
# -----------------------------
y = df['Cars Prices']
X = df.drop('Cars Prices', axis=1)

# For now: keep only numeric features
X = X.select_dtypes(include=[np.number])

# -----------------------------
# 5. Train / Val / Test Split
# -----------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

# -----------------------------
# 6. Standardization
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# -----------------------------
# 7. To PyTorch Tensors
# -----------------------------
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_val   = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
y_test  = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# -----------------------------
# 8. MLP Model
# -----------------------------
class CarPricingMLP(nn.Module):
    def __init__(self, input_dim):
        super(CarPricingMLP, self).__init__()
        self.hidden1 = nn.Linear(input_dim, 64)
        self.hidden2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        x = self.output(x)
        return x

model = CarPricingMLP(X_train.shape[1])
print(model)


print("Dataset shape:", df.shape)
print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
print(X.columns)

print("\nTrain samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Test samples:", X_test.shape[0])


print("\nAfter scaling:")
print("X_train mean:", X_train.mean().item())
print("X_train std:", X_train.std().item())
print("\nSample features (first 5 rows):")
print(X_train[:5])

CarPricingMLP(
  (hidden1): Linear(in_features=1, out_features=64, bias=True)
  (hidden2): Linear(in_features=64, out_features=32, bias=True)
  (output): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)
Dataset shape: (1218, 11)
Features shape: (1218, 1)
Target shape: (1218,)

Feature columns:
Index(['Seats'], dtype='object')

Train samples: 852
Validation samples: 183
Test samples: 183

After scaling:
X_train mean: -1.6790040469061296e-09
X_train std: 1.0005873441696167

Sample features (first 5 rows):
tensor([[ 0.1093],
        [ 0.7563],
        [ 0.1093],
        [-1.8315],
        [ 0.1093]])


## Motivation for the Second Preprocessing Pipeline

In the initial preprocessing pipeline, only numerical features were selected. 
As a result, the model received a single input feature (`in_features = 1`), which 
significantly limited the amount of information available to the neural network.

This naive representation caused the model to learn a function of the form:

f(x) = price

which is insufficient for a complex real-world problem such as car price prediction.

In the second pipeline, categorical features (such as manufacturer, engine type, 
and fuel type) were transformed using One-Hot Encoding. This conversion increased 
the input dimensionality to 2937 features, enabling the model to learn a mapping of 
the form:

f(x₁, x₂, ..., x₂₉₃₇) = price

This demonstrates that model performance is primarily constrained by data 
representation rather than network architecture. The second pipeline provides a 
richer and more realistic feature space, allowing the MLP to approximate the 
true underlying relationship between car attributes and price.


In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import re

# -----------------------------
# 1. Load Dataset
# -----------------------------
df = pd.read_csv('CarsDatasets.csv')

# -----------------------------
# 2. Clean Price Column
# -----------------------------
def clean_price(price):
    if pd.isna(price):
        return np.nan
    
    numbers = re.findall(r'\d+', price)
    
    if len(numbers) == 0:
        return np.nan
    
    numbers = [float(n) for n in numbers]
    return np.mean(numbers)

df['Cars Prices'] = df['Cars Prices'].apply(clean_price)

# -----------------------------
# 3. Basic Preprocessing
# -----------------------------
df = df.fillna(df.median(numeric_only=True))

# -----------------------------
# 4. Split Features and Target
# -----------------------------
y = df['Cars Prices']
X = df.drop('Cars Prices', axis=1)

# -----------------------------
# 5. One-Hot Encoding (KEY FIX)
# -----------------------------
X = pd.get_dummies(X)

print("Number of input features after encoding:", X.shape[1])

# -----------------------------
# 6. Train / Val / Test Split
# -----------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

# -----------------------------
# 7. Standardization
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# -----------------------------
# 8. To PyTorch Tensors
# -----------------------------
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val,   dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_val   = torch.tensor(y_val.values,   dtype=torch.float32).view(-1, 1)
y_test  = torch.tensor(y_test.values,  dtype=torch.float32).view(-1, 1)

# -----------------------------
# 9. MLP Model
# -----------------------------
class CarPricingMLP(nn.Module):
    def __init__(self, input_dim):
        super(CarPricingMLP, self).__init__()
        self.hidden1 = nn.Linear(input_dim, 64)
        self.hidden2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        x = self.output(x)
        return x

model = CarPricingMLP(X_train.shape[1])
print(model)


print("Dataset shape:", df.shape)
print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
print(X.columns)

print("\nTrain samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Test samples:", X_test.shape[0])


print("\nAfter scaling:")
print("X_train mean:", X_train.mean().item())
print("X_train std:", X_train.std().item())
print("\nSample features (first 5 rows):")
print(X_train[:5])


Number of input features after encoding: 2937
CarPricingMLP(
  (hidden1): Linear(in_features=2937, out_features=64, bias=True)
  (hidden2): Linear(in_features=64, out_features=32, bias=True)
  (output): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)
Dataset shape: (1218, 11)
Features shape: (1218, 2937)
Target shape: (1218,)

Feature columns:
Index(['Seats', 'Company Names_ASTON MARTIN', 'Company Names_AUDI',
       'Company Names_Acura', 'Company Names_BENTLEY', 'Company Names_BMW',
       'Company Names_Bugatti', 'Company Names_Cadillac',
       'Company Names_Chevrolet', 'Company Names_FERRARI',
       ...
       'Torque_881 Nm', 'Torque_885 Nm', 'Torque_893 Nm', 'Torque_90 Nm',
       'Torque_900 Nm', 'Torque_93 Nm', 'Torque_94 Nm', 'Torque_95 Nm',
       'Torque_950 Nm', 'Torque_967 Nm'],
      dtype='object', length=2937)

Train samples: 852
Validation samples: 183
Test samples: 183

After scaling:
X_train mean: 1.4905424672306822e-09
X_train std: 0.88532102

## Section 2: Comparison of Nominal Feature Encoding Methods
In this section, we evaluate three different techniques for handling categorical data:
    One-Hot Encoding: Representing each category as a binary vector.
    Target Encoding: Replacing categories with the mean of the target variable (Price).
    Layer Embedding: Using a learnable dense vector representation for each category.

We will analyze their impact on RMSE (Root Mean Squared Error) and Convergence Speed.

### 1. Target Encoding Implementation
Target encoding replaces each label with the average price of cars in that category.

### 2. Layer Embedding Architecture
For embeddings, we treat each categorical feature as an index and pass it through an nn.Embedding layer. This allows the model to "learn" the relationship between categories

In [11]:
# Function for manual Target Encoding to avoid data leakage
def apply_target_encoding(X_train, y_train, X_val, X_test, categorical_cols):
    X_train_enc = X_train.copy()
    X_val_enc = X_val.copy()
    X_test_enc = X_test.copy()
    
    for col in categorical_cols:
        # Calculate mean price for each category in TRAIN set only
        target_mean = y_train.groupby(X_train[col]).mean()
        
        # Map values to train, val, and test
        X_train_enc[col] = X_train[col].map(target_mean)
        X_val_enc[col] = X_val[col].map(target_mean)
        X_test_enc[col] = X_test[col].map(target_mean)
        
        # Fill NaN (categories in val/test not seen in train) with global mean
        global_mean = y_train.mean()
        X_train_enc[col].fillna(global_mean, inplace=True)
        X_val_enc[col].fillna(global_mean, inplace=True)
        X_test_enc[col].fillna(global_mean, inplace=True)
        
    return X_train_enc, X_val_enc, X_test_enc


In [12]:
class MLPWithEmbeddings(nn.Module):
    def __init__(self, embedding_dims, num_numerical_cols):
        super(MLPWithEmbeddings, self).__init__()
        
        # Create embedding layers for each categorical feature
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_categories, emb_dim) 
            for num_categories, emb_dim in embedding_dims
        ])
        
        # Calculate total input size (sum of all embedding dims + numerical dims)
        total_emb_dim = sum(emb_dim for _, emb_dim in embedding_dims)
        input_dim = total_emb_dim + num_numerical_cols
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x_cat, x_num):
        # Pass each categorical feature through its respective embedding layer
        emb_outputs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(emb_outputs + [x_num], dim=1)
        return self.network(x)

Data Representation
        ↓
Same MLP Architecture
        ↓
Train for N epochs
        ↓
Store RMSE(train), RMSE(val)
        ↓
Plot curves


In [13]:
from sklearn.metrics import mean_squared_error
import math

def train_model(model, X_train, y_train, X_val, y_val, epochs=50, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_rmse = []
    val_rmse = []
    
    for epoch in range(epochs):
        # ---- TRAIN ----
        model.train()
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        
        rmse_train = math.sqrt(loss.item())
        
        # ---- VALIDATION ----
        model.eval()
        with torch.no_grad():
            y_val_pred = model(X_val)
            val_loss = criterion(y_val_pred, y_val)
            rmse_val = math.sqrt(val_loss.item())
        
        train_rmse.append(rmse_train)
        val_rmse.append(rmse_val)
        
    return train_rmse, val_rmse


### 1. One-Hot

In [14]:
X = pd.get_dummies(X)

model_oh = CarPricingMLP(X_train.shape[1])
rmse_train_oh, rmse_val_oh = train_model(
    model_oh, X_train, y_train, X_val, y_val
)

### 2. Target Encoding

In [15]:
categorical_cols = [
    'Company Names',
    'Cars Names',
    'Engines',
    'Fuel Types'
]

X_train_te, X_val_te, X_test_te = apply_target_encoding(
    X_train_raw, y_train_raw,
    X_val_raw, X_test_raw,
    categorical_cols
)

model_te = CarPricingMLP(X_train_te.shape[1])
rmse_train_te, rmse_val_te = train_model(
    model_te, X_train_te, y_train, X_val_te, y_val
)


NameError: name 'X_train_raw' is not defined

### 3. Embedding

In [16]:
from sklearn.preprocessing import LabelEncoder

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_val[col]   = le.transform(X_val[col])
    X_test[col]  = le.transform(X_test[col])
    encoders[col] = le

embedding_dims = []
for col in categorical_cols:
    n_categories = X_train[col].nunique()
    emb_dim = min(50, n_categories // 2)
    embedding_dims.append((n_categories, emb_dim))

X_train_cat = torch.tensor(X_train[categorical_cols].values, dtype=torch.long)
X_train_num = torch.tensor(X_train[numerical_cols].values, dtype=torch.float32)

model_emb = MLPWithEmbeddings(embedding_dims, len(numerical_cols))
rmse_train_emb, rmse_val_emb = train_model(
    model_emb,
    (X_train_cat, X_train_num),
    y_train,
    (X_val_cat, X_val_num),
    y_val
)


IndexError: too many indices for tensor of dimension 2